In [40]:
import networkx as nx
import itertools

from bp.world import random_grid_world, Scenario

demands = {
    ((1, 0), (1, 3)): 6,
    ((0, 0), (2, 3)): 6,
}

world = random_grid_world(
    rows=4,
    cols=4,
    demands=demands,
    seed=0,
)
G = world.network.graph
world.network.bpr_beta = 1
nominal = Scenario.from_world("nominal", world)

print(f"{world.total_population=}")
for od, n_k in demands.items():
    print(f"Demand for {od}: {n_k}")

world.total_population=12
Demand for ((1, 0), (1, 3)): 6
Demand for ((0, 0), (2, 3)): 6


In [41]:
accident_congestion = 10
target_edge = ((1, 1), (1, 2))

travel_time = dict(nominal.travel_time)
travel_time[target_edge] += accident_congestion

accident = Scenario(
    name="accident",
    travel_time=travel_time,
    discomfort=nominal.discomfort,
    hazard=nominal.hazard,
    cost=nominal.cost,
    emissions=nominal.emissions,
    policing=nominal.policing
)

scenarios = {
    "nominal": (nominal, .8),
    "accident": (accident, .2),
}

assert all(prior >= 0 for _, prior in scenarios.values()), "invalid prior distribution"
assert sum(prior for _, prior in scenarios.values()) == 1, "invalid prior distribution"

In [42]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("asymmetric dictator (anonymous)")
model.setParam("OutputFlag", 0)

V = world.ordered_nodes
A = world.ordered_arcs
I = world.I
N = world.individuals

n = world.total_population
t = world.network.travel_time
c = world.network.capacity
alpha = world.network.bpr_alpha
beta = world.network.bpr_beta

In [43]:
from collections.abc import Mapping, Sequence

from bp.world import Arc, Node

def edge_path(path: Sequence[Node]) -> Sequence[Arc]:
    return list(itertools.pairwise(path))

In [44]:
# reminder: beta=1 is fixed
# tau[omega, a, k] := average cost for k players on arc a under state omega
tau = {}
for scenario_name, (omega, _) in scenarios.items():
    for a in A:
        for k in range(n + 1):
            tau[scenario_name, a, k] = omega.travel_time[a] * (1 + alpha * ((k - 1) / c[a]) ** beta)

In [ ]:
# decision: x_od[omega, od, a] := total flow on arc a under scenario omega for a given OD pair
x_od = {
    (scenario_name, od, a): model.addVar(vtype=GRB.INTEGER, lb=0, ub=demand, name=f"x_od_{scenario_name}_{od}_{a}")
    for od, demand in demands.items()
    for a in A
    for scenario_name in scenarios
}

# decision: x[omega, a] := total flow on arc a under scenario omega
x = {
    (scenario_name, a): model.addVar(vtype=GRB.INTEGER, lb=0, ub=n, name=f"x_{scenario_name}_{a}")
    for a in A
    for scenario_name in scenarios
}

# constraint: flow conservation
for scenario_name in scenarios:
    for od, demand in demands.items():
        for v in V:
            flow_out = gp.quicksum(x_od[scenario_name, od, a] for a in A if a[0] == v)
            flow_in = gp.quicksum(x_od[scenario_name, od, a] for a in A if a[1] == v)
            flow = demand if v == od[0] else (-demand if v == od[1] else 0)

            model.addConstr(flow_out - flow_in == flow, name=f"flow_{scenario_name}_{od}_{v}")

# constraint: x[omega, a] = \sum_{t \in T} x_od[omega, t, a]
for scenario_name in scenarios:
    for a in A:
        model.addConstr(
            x[scenario_name, a] == gp.quicksum(x_od[scenario_name, od, a] for od in demands),
            name=f"x_{a}"
        )

# objective: \min \sum_{\omega \in \Omega} \mu_\omega \sum_{a \in A} x[\omega, a] \cdot tau[\omega, a, x[a]]
for scenario_name, (_, mu) in scenarios.items():
    for a in A:
        model.setPWLObj(
            x[scenario_name, a],
            list(range(n + 1)),
            [mu * k * tau[scenario_name, a, k] for k in range(n + 1)]
        )

In [46]:
model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

In [47]:
def debug():
    print(f"{model.ObjVal=:.2f}")
    print(f"{model.Runtime=:.3f}")

    for scenario_name in scenarios:
        print(f"\nScenario: {scenario_name}")
        for od, demand in demands.items():
            print(f"\t{od}:")

            flows = {a: flow for a in A if (flow:=x_od[scenario_name, od, a].X) > 0}

            # greedily decompose into paths
            for _ in range(demand):
                path = [od[0]]
                curr = od[0]

                while curr != od[1]: # NOTE: assuming flow conservation from above
                    for a, flow in flows.items():
                        if a[0] == curr and flow > 0:
                            if flow > 1:
                                flows[a] -= 1
                            else:
                                del flows[a]
                            curr = a[1]
                            path.append(curr)
                            break

                print(f"\t\t{path}")

print("OPTIMAL")
optimal_social_cost = model.ObjVal
print(f"{optimal_social_cost=:.2f}")
debug()

OPTIMAL
optimal_social_cost=350.21
model.ObjVal=350.21
model.Runtime=0.005

Scenario: nominal
	((1, 0), (1, 3)):
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
	((0, 0), (2, 3)):
		[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
		[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
		[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3)]
		[(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3)]
		[(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3)]
		[(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3)]

Scenario: accident
	((1, 0), (1, 3)):
		[(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (1, 3)]
	((0, 0), (2, 3)):
		[(

In [51]:
# objective: TODO
for scenario_name, (_, mu) in scenarios.items():
    for a in A:
        model.setPWLObj(
            x[scenario_name, a],
            list(range(n + 1)),
            [
                mu * sum(
                    tau[scenario_name, a, k]
                    for k in range(1, x + 1)
                ) for x in range(n + 1)
            ]
        )

model.optimize()
assert model.Status == GRB.OPTIMAL, f"Optimization failed: {model.Status}"

selfish_social_cost = 0
for scenario_name, (_, mu) in scenarios.items():
    for a in A:
        k = x[scenario_name, a].X
        selfish_social_cost += mu * k * tau[scenario_name, a, k]

print("SELFISH")
PoA = selfish_social_cost / optimal_social_cost
print(f"{selfish_social_cost=:.2f}")
print(f"{PoA=:.5f}")
debug()

SELFISH
selfish_social_cost=351.60
PoA=1.00397
model.ObjVal=329.06
model.Runtime=0.004

Scenario: nominal
	((1, 0), (1, 3)):
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (1, 2), (1, 3)]
	((0, 0), (2, 3)):
		[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
		[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
		[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
		[(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3)]
		[(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3)]
		[(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3)]

Scenario: accident
	((1, 0), (1, 3)):
		[(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (0, 1), (0, 2), (0, 3), (1, 3)]
		[(1, 0), (1, 1), (2, 1), (2, 2), (1, 2), (1, 3)]
		[(1, 0), (1, 1), (2, 1), (2, 2), (1, 2), (1, 3)]
		[(1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (1,